<a href="https://colab.research.google.com/github/24071a6236-jpg/polymathai/blob/main/Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Install required packages if needed
!pip -q install datasets scikit-learn pandas numpy

In [15]:
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA

import os
import pickle

print("Libraries imported successfully.")

Libraries imported successfully.


In [16]:
# Final seeds specified by the project protocol
SEEDS = [42, 1337, 2024]

print("Seeds:", SEEDS)

Seeds: [42, 1337, 2024]


In [17]:
toniots = load_dataset("codymlewis/TON_IoT_network")

ton_train_df = toniots["train"].to_pandas()
ton_test_df = toniots["test"].to_pandas()

print("ToN-IoT train shape:", ton_train_df.shape)
print("ToN-IoT test shape :", ton_test_df.shape)

print("\nToN-IoT columns:")
print(ton_train_df.columns.tolist())

ToN-IoT train shape: (211043, 44)
ToN-IoT test shape : (211043, 44)

ToN-IoT columns:
['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type']


In [18]:
nsl_kdd = load_dataset("Mireu-Lab/NSL-KDD")

nsl_train_df = nsl_kdd["train"].to_pandas()
nsl_test_df = nsl_kdd["test"].to_pandas()

print("NSL-KDD train shape:", nsl_train_df.shape)
print("NSL-KDD test shape :", nsl_test_df.shape)

print("\nNSL-KDD columns:")
print(nsl_train_df.columns.tolist())

NSL-KDD train shape: (151165, 42)
NSL-KDD test shape : (34394, 42)

NSL-KDD columns:
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class']


In [19]:
print("ToN-IoT target values:")
print(ton_train_df["label"].value_counts())

print("\nNSL-KDD target values:")
print(nsl_train_df["class"].value_counts())

ToN-IoT target values:
label
1    161043
0     50000
Name: count, dtype: int64

NSL-KDD target values:
class
normal     80792
anomaly    70373
Name: count, dtype: int64


In [20]:
def preprocess_dataset(df, target_column, seeds):

    # Separate features and target
    X = df.drop(columns=[target_column]).copy()
    y = df[target_column].copy()

    # Remove attack-type information if present
    # This prevents target leakage in ToN-IoT.
    if "type" in X.columns:
        X = X.drop(columns=["type"])

    results = {}

    for seed in seeds:

        print("=" * 60)
        print(f"Processing seed: {seed}")

        # --------------------------------------------------
        # 1. 70/30 stratified split
        # --------------------------------------------------
        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y,
            test_size=0.30,
            stratify=y,
            random_state=seed
        )

        # --------------------------------------------------
        # 2. Identify categorical columns
        # --------------------------------------------------
        categorical_cols = X_train.select_dtypes(
            include=["object"]
        ).columns.tolist()

        # --------------------------------------------------
        # 3. Ordinal Encoding
        # --------------------------------------------------
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

        X_train_enc = X_train.copy()
        X_val_enc = X_val.copy()

        if len(categorical_cols) > 0:
            X_train_enc[categorical_cols] = encoder.fit_transform(
                X_train[categorical_cols]
            )

            X_val_enc[categorical_cols] = encoder.transform(
                X_val[categorical_cols]
            )

        # --------------------------------------------------
        # 4. Standard Scaling
        # --------------------------------------------------
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train_enc)
        X_val_scaled = scaler.transform(X_val_enc)

        # --------------------------------------------------
        # 5. PCA → exactly 8 components
        # --------------------------------------------------
        pca = PCA(n_components=8)

        X_train_pca = pca.fit_transform(X_train_scaled)
        X_val_pca = pca.transform(X_val_scaled)

        # --------------------------------------------------
        # Save results for this seed
        # --------------------------------------------------
        results[seed] = {
            "X_train": X_train_pca,
            "X_val": X_val_pca,
            "y_train": y_train.to_numpy(),
            "y_val": y_val.to_numpy(),
            "encoder": encoder,
            "scaler": scaler,
            "pca": pca,
            "categorical_cols": categorical_cols
        }

        print("Train shape:", X_train_pca.shape)
        print("Validation shape:", X_val_pca.shape)
        print(
            "PCA explained variance:",
            round(pca.explained_variance_ratio_.sum(), 4)
        )

    return results

In [23]:
nsl_results = preprocess_dataset(
    nsl_train_df,
    target_column="class",
    seeds=SEEDS
)

Processing seed: 42
Train shape: (105815, 8)
Validation shape: (45350, 8)
PCA explained variance: 0.606
Processing seed: 1337
Train shape: (105815, 8)
Validation shape: (45350, 8)
PCA explained variance: 0.6252
Processing seed: 2024
Train shape: (105815, 8)
Validation shape: (45350, 8)
PCA explained variance: 0.623


In [25]:
# ============================================
# MILESTONE 3 - RESET + LOAD ToN-IoT
# ============================================

from datasets import load_dataset
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.decomposition import PCA

# Required seeds
SEEDS = [42, 1337, 2024]

# Load ToN-IoT
toniot = load_dataset("codymlewis/TON_IoT_network")

ton_train_df = toniot["train"].to_pandas()
ton_test_df = toniot["test"].to_pandas()

print("ToN-IoT loaded successfully")
print("Train:", ton_train_df.shape)
print("Test :", ton_test_df.shape)
print("Seeds:", SEEDS)

ToN-IoT loaded successfully
Train: (211043, 44)
Test : (211043, 44)
Seeds: [42, 1337, 2024]


In [26]:
# ============================================
# ToN-IoT PREPROCESSING FUNCTION
# ============================================

def preprocess_dataset(df, target_column, seeds):

    X = df.drop(columns=[target_column]).copy()
    y = df[target_column].copy()

    # Remove type column to prevent target leakage
    if "type" in X.columns:
        X = X.drop(columns=["type"])

    results = {}

    for seed in seeds:

        print(f"\nProcessing seed {seed}...")

        # 70/30 stratified split
        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y,
            test_size=0.30,
            stratify=y,
            random_state=seed
        )

        # Find categorical columns
        categorical_cols = X_train.select_dtypes(
            include=["object"]
        ).columns.tolist()

        print("Categorical columns:", len(categorical_cols))

        # Ordinal encoding
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )

        X_train_enc = X_train.copy()
        X_val_enc = X_val.copy()

        if len(categorical_cols) > 0:

            X_train_enc[categorical_cols] = encoder.fit_transform(
                X_train[categorical_cols]
            )

            X_val_enc[categorical_cols] = encoder.transform(
                X_val[categorical_cols]
            )

        # StandardScaler
        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train_enc)
        X_val_scaled = scaler.transform(X_val_enc)

        # PCA → exactly 8 components
        pca = PCA(
            n_components=8,
            random_state=seed
        )

        X_train_pca = pca.fit_transform(X_train_scaled)
        X_val_pca = pca.transform(X_val_scaled)

        results[seed] = {
            "X_train": X_train_pca,
            "X_val": X_val_pca,
            "y_train": np.asarray(y_train),
            "y_val": np.asarray(y_val),
            "encoder": encoder,
            "scaler": scaler,
            "pca": pca,
            "categorical_cols": categorical_cols
        }

        print("X_train:", X_train_pca.shape)
        print("X_val  :", X_val_pca.shape)
        print(
            "PCA variance:",
            round(pca.explained_variance_ratio_.sum(), 4)
        )

    return results

In [27]:
# ============================================
# PROCESS ToN-IoT
# ============================================

ton_results = preprocess_dataset(
    ton_train_df,
    target_column="label",
    seeds=SEEDS
)

print("\n================================")
print("ToN-IoT preprocessing COMPLETE")
print("================================")


Processing seed 42...
Categorical columns: 26
X_train: (147730, 8)
X_val  : (63313, 8)
PCA variance: 0.6095

Processing seed 1337...
Categorical columns: 26
X_train: (147730, 8)
X_val  : (63313, 8)
PCA variance: 0.6092

Processing seed 2024...
Categorical columns: 26
X_train: (147730, 8)
X_val  : (63313, 8)
PCA variance: 0.6053

ToN-IoT preprocessing COMPLETE


In [28]:
for seed in SEEDS:

    data = ton_results[seed]

    print(f"\n========== Seed {seed} ==========")
    print("X_train:", data["X_train"].shape)
    print("X_val  :", data["X_val"].shape)
    print("y_train:", data["y_train"].shape)
    print("y_val  :", data["y_val"].shape)

    assert data["X_train"].shape[1] == 8
    assert data["X_val"].shape[1] == 8

print("\n✅ ToN-IoT Milestone 3 preprocessing verified.")


========== Seed 42 ==========
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)

========== Seed 1337 ==========
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)

========== Seed 2024 ==========
X_train: (147730, 8)
X_val  : (63313, 8)
y_train: (147730,)
y_val  : (63313,)

✅ ToN-IoT Milestone 3 preprocessing verified.


In [29]:
# ============================================
# LOAD NSL-KDD
# ============================================

nsl = load_dataset("Mireu-Lab/NSL-KDD")

nsl_train_df = nsl["train"].to_pandas()
nsl_test_df = nsl["test"].to_pandas()

print("NSL-KDD loaded successfully")
print("Train shape:", nsl_train_df.shape)
print("Test shape :", nsl_test_df.shape)

print("\nColumns:")
print(nsl_train_df.columns.tolist())

NSL-KDD loaded successfully
Train shape: (151165, 42)
Test shape : (34394, 42)

Columns:
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class']


In [30]:
# ============================================
# PROCESS NSL-KDD
# ============================================

nsl_results = preprocess_dataset(
    nsl_train_df,
    target_column="class",
    seeds=SEEDS
)

print("\n================================")
print("NSL-KDD preprocessing COMPLETE")
print("================================")


Processing seed 42...
Categorical columns: 3
X_train: (105815, 8)
X_val  : (45350, 8)
PCA variance: 0.606

Processing seed 1337...
Categorical columns: 3
X_train: (105815, 8)
X_val  : (45350, 8)
PCA variance: 0.6252

Processing seed 2024...
Categorical columns: 3
X_train: (105815, 8)
X_val  : (45350, 8)
PCA variance: 0.623

NSL-KDD preprocessing COMPLETE


In [31]:
# ============================================
# VERIFY NSL-KDD
# ============================================

for seed in SEEDS:

    data = nsl_results[seed]

    print(f"\n========== Seed {seed} ==========")
    print("X_train:", data["X_train"].shape)
    print("X_val  :", data["X_val"].shape)
    print("y_train:", data["y_train"].shape)
    print("y_val  :", data["y_val"].shape)

    assert data["X_train"].shape[1] == 8
    assert data["X_val"].shape[1] == 8

print("\n✅ NSL-KDD preprocessing verified.")


========== Seed 42 ==========
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (45350,)

========== Seed 1337 ==========
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (45350,)

========== Seed 2024 ==========
X_train: (105815, 8)
X_val  : (45350, 8)
y_train: (105815,)
y_val  : (45350,)

✅ NSL-KDD preprocessing verified.
